# 02. Preprocessing & Feature Engineering Pipeline (Role 3)
Demonstrasi pipeline anti-leakage untuk Skenario 2 (`ui10`) dan Skenario 1 (`full`) menggunakan modul `src/features.py` dan `src/preprocessing.py`.

In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import FEATURE_NAMES, RANDOM_STATE, TARGET, TEST_SIZE
from src.preprocessing import build_preprocessor

df = pd.read_csv(ROOT_DIR / "data" / "raw" / "train.csv")
print("Shape dataset mentah:", df.shape)

Shape dataset mentah: (1460, 81)


## 1. Train-Test Split di Awal (Anti Data Leakage)
Pembagian data dilakukan sebelum proses `fit()` apa pun. Seluruh parameter imputasi dan scaling hanya dipelajari dari `X_train`.

In [2]:
X = df.drop(columns=[TARGET])
y = np.log1p(df[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

X_train: (1168, 80), X_test: (292, 80)


## 2. Demonstrasi Skenario 2 (`ui10`) — Model Produksi
Menguji `build_preprocessor(feature_set="ui10")` untuk model tree (`for_linear=False`) dan model linear (`for_linear=True`).

In [3]:
for for_linear in (False, True):
    prep_ui10 = build_preprocessor(feature_set="ui10", for_linear=for_linear)
    X_train_ui10 = prep_ui10.fit_transform(X_train[FEATURE_NAMES])
    X_test_ui10 = prep_ui10.transform(X_test[FEATURE_NAMES])
    feat_names_ui10 = prep_ui10.get_feature_names_out()

    assert not np.isnan(X_train_ui10).any(), "Ditemukan NaN pada X_train_ui10!"
    assert not np.isnan(X_test_ui10).any(), "Ditemukan NaN pada X_test_ui10!"

    print(f"=== ui10 (for_linear={for_linear}) ===")
    print("Shape train:", X_train_ui10.shape, "| Shape test:", X_test_ui10.shape)
    print("Jumlah fitur output:", len(feat_names_ui10))
    print("Contoh nama fitur:", feat_names_ui10[:12].tolist(), "\n")

=== ui10 (for_linear=False) ===
Shape train: (1168, 35) | Shape test: (292, 35)
Jumlah fitur output: 35
Contoh nama fitur: ['GrLivArea', 'TotalBsmtSF', 'LotArea', 'TotalSF', 'OverallQual', 'GarageCars', 'FullBath', 'Fireplaces', 'HouseAge', 'KitchenQual', 'Neighborhood_Blmngtn', 'Neighborhood_Blueste'] 

=== ui10 (for_linear=True) ===
Shape train: (1168, 35) | Shape test: (292, 35)
Jumlah fitur output: 35
Contoh nama fitur: ['GrLivArea', 'TotalBsmtSF', 'LotArea', 'TotalSF', 'OverallQual', 'GarageCars', 'FullBath', 'Fireplaces', 'HouseAge', 'KitchenQual', 'Neighborhood_Blmngtn', 'Neighborhood_Blueste'] 



## 3. Demonstrasi Skenario 1 (`full`) — Benchmark Fitur Penuh
Menguji `build_preprocessor(feature_set="full")` dengan penanganan `NA` bermakna (`'None'` / `0`) vs missing murni (`median` / `modus`).

In [4]:
for for_linear in (False, True):
    prep_full = build_preprocessor(feature_set="full", for_linear=for_linear)
    X_train_full = prep_full.fit_transform(X_train)
    X_test_full = prep_full.transform(X_test)
    feat_names_full = prep_full.get_feature_names_out()

    assert not np.isnan(X_train_full).any(), "Ditemukan NaN pada X_train_full!"
    assert not np.isnan(X_test_full).any(), "Ditemukan NaN pada X_test_full!"

    print(f"=== full (for_linear={for_linear}) ===")
    print("Shape train:", X_train_full.shape, "| Shape test:", X_test_full.shape)
    print("Jumlah fitur output:", len(feat_names_full))

=== full (for_linear=False) ===
Shape train: (1168, 260) | Shape test: (292, 260)
Jumlah fitur output: 260


=== full (for_linear=True) ===
Shape train: (1168, 260) | Shape test: (292, 260)
Jumlah fitur output: 260
